# Langextract Functions Review

## `langextract.extract()` review

In [ ]:
import os
openai_api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
%load_ext rich

In [ ]:
import warnings

import langextract as lx

from langextract import annotation
from langextract import data
from langextract import factory
from langextract import prompting
from langextract import resolver, chunking, exceptions
from collections.abc import Iterable, Iterator, Sequence


In [ ]:
debug = True
max_workers = 16
batch_length = 16

In [ ]:
prompt_description = """
    Sos un asistente especializado en el análisis de documentos judiciales.
    Tu tarea es identificar y extraer menciones de información sensible para su posterior anonimización.
    Debés detectar fragmentos textuales que correspondan a cualquiera de las siguientes entidades:

    - "BANCO": Nombre de una entidad bancaria, pública o privada.
    - "CBU": Código Bancario Uniforme (22 dígitos) de una cuenta.
    - "CORREO_ELECTRONICO": Dirección de correo electrónico.
    - "CUIJ": Código Único de Identificación Jurídica de causas judiciales.
    - "CUIT_CUIL": Número de CUIT o CUIL de una persona física o jurídica.
    - "DIRECCION": Dirección postal específica (calle, número, etc.).
    - "DNI": Número de Documento Nacional de Identidad u otro documento identificatorio.
    - "EDAD": Edad explícita de una persona.
    - "ESTUDIOS": Nivel o institución educativa que permita identificar a la persona (ej. "primario incompleto", "secundario completo", "Licenciado en…").
    - "FECHA": Fecha completa o parcial (día, mes y/o año).
    - "LINK": Enlace o URL a una página web.
    - "LOC": Localización geográfica específica (ciudad, barrio, comisaría, etc.).
    - "MARCA_AUTOMOVIL": Marca de un vehículo (ej. Toyota, Ford).
    - "NACIONALIDAD": Nacionalidad de una persona (ej. "argentino", "brasileña").
    - "NUM_ACTUACION": Número identificatorio de una actuación administrativa o contravencional.
    - "NUM_CAJA_AHORRO": Número completo de una caja de ahorro o cuenta bancaria.
    - "NUM_EXPEDIENTE": Número de expediente judicial o administrativo.
    - "NUM_MATRICULA": Número de matrícula profesional o académica.
    - "PATENTE_DOMINIO": Patente o dominio de un vehículo.
    - "PER": Nombre y apellido(s) de una persona física. Los nombres inicializados y los apodos también cuentan como información sensible a anonimizar.
    - "TELEFONO": Número telefónico (fijo o celular).
"""

In [ ]:
examples = [
    lx.data.ExampleData(
        text="El 5 de mayo de 2023 el señor Fiscal indicó que realizó distintas medidas de prueba y que del resultado surge que tanto la investigada como el menor Juan Pérez se domicilian en la calle Sarmiento 1234, de la localidad de Moreno, por lo que solicitó que se declare la incompetencia en razón del territorio y se envíe el caso al Juzgado de Garantías que corresponda del Departamento Judicial de Moreno, con jurisdicción en el partido de Moreno.",
        extractions=[
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="5 de mayo de 2023"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan Pérez"
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION", extraction_text="Sarmiento 1234"
            ),
            lx.data.Extraction(
                extraction_class="LOC", extraction_text="Moreno"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="JUZGADO DE 1RA INSTANCIA EN LO PENAL CONTRAVENCIONAL Y DE FALTAS N°10 SECRETARIA N°19\nCarlos Gómez sobre 84 - HOMICIDIO CULPOSO Y OTROS\nNúmero: 52345/2022\nCUIJ: 12-34567890-1\nActuación Nro: 2022-009876",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Carlos Gómez"
            ),
            lx.data.Extraction(
                extraction_class="NUM_EXPEDIENTE", extraction_text="52345/2022"
            ),
            lx.data.Extraction(
                extraction_class="CUIJ", extraction_text="12-34567890-1"
            ),
            lx.data.Extraction(
                extraction_class="NUM_ACTUACION", extraction_text="2022-009876"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Acusado: Miguel Torres, DNI 30123456, nacido el 14/02/1990, de 34 años de edad, de nacionalidad paraguaya, género varón cis, con estudios secundarios completos, hizo hasta 3er año porque fue padre joven, con último domicilio en Av. Corrientes 3456, de esta ciudad, donde vive con su hermana y su cuñado Jorge Pérez. Tiene dos hijos a su cargo, de 5 y 8 años. Su hijo de 8 vive con él, su hija de 5 vive con su madre, Laura Fernández.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Miguel Torres"
            ),
            lx.data.Extraction(
                extraction_class="DNI", extraction_text="30123456"
            ),
            lx.data.Extraction(
                extraction_class="FECHA", extraction_text="14/02/1990"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="34"),
            lx.data.Extraction(
                extraction_class="NACIONALIDAD", extraction_text="paraguaya"
            ),
            lx.data.Extraction(
                extraction_class="ESTUDIOS",
                extraction_text="estudios secundarios completos",
            ),
            lx.data.Extraction(
                extraction_class="DIRECCION",
                extraction_text="Av. Corrientes 3456",
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Jorge Pérez"
            ),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="5"),
            lx.data.Extraction(extraction_class="EDAD", extraction_text="8"),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Laura Fernández"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="El testigo Juan López dejó asentado su número de contacto: 11-2345-6789. Indicó que la médica Dra. Ana García, MN 12345, asistió al lugar donde se hallaba un vehículo Volkswagen, patente AB123CD.",
        extractions=[
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Juan López"
            ),
            lx.data.Extraction(
                extraction_class="TELEFONO", extraction_text="11-2345-6789"
            ),
            lx.data.Extraction(
                extraction_class="PER", extraction_text="Ana García"
            ),
            lx.data.Extraction(
                extraction_class="NUM_MATRICULA", extraction_text="12345"
            ),
            lx.data.Extraction(
                extraction_class="MARCA_AUTOMOVIL",
                extraction_text="Volkswagen",
            ),
            lx.data.Extraction(
                extraction_class="PATENTE_DOMINIO", extraction_text="AB123CD"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Se identificó una transferencia bancaria con los siguientes datos: CUIT 20-12345678-3, CBU 2850590940090412345671, Caja de Ahorro N° 12345678, Banco Nación.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CUIT_CUIL", extraction_text="20-12345678-3"
            ),
            lx.data.Extraction(
                extraction_class="CBU",
                extraction_text="2850590940090412345671",
            ),
            lx.data.Extraction(
                extraction_class="NUM_CAJA_AHORRO", extraction_text="12345678"
            ),
            lx.data.Extraction(
                extraction_class="BANCO", extraction_text="Banco Nación"
            ),
        ],
    ),
    lx.data.ExampleData(
        text="Para mayor información, comunicarse a fiscalia.central@justicia.gob.ar o visitar el sitio https://justicia.gob.ar/actuaciones.",
        extractions=[
            lx.data.Extraction(
                extraction_class="CORREO_ELECTRONICO",
                extraction_text="fiscalia.central@justicia.gob.ar",
            ),
            lx.data.Extraction(
                extraction_class="LINK",
                extraction_text="https://justicia.gob.ar/actuaciones",
            ),
        ],
    ),
]

In [ ]:
# Examples assertion
if not examples:
    raise ValueError(
        "Examples are required for reliable extraction. Please provide at least"
        " one ExampleData object with sample extractions."
    )

In [ ]:
# Debug and max_workers configuration
if debug:
    # pylint: disable=import-outside-toplevel
    from langextract import debug_utils

    debug_utils.configure_debug_logging()

if max_workers is not None and batch_length < max_workers:
    warnings.warn(
        f"batch_length ({batch_length}) < max_workers ({max_workers}). "
        f"Only {batch_length} workers will be used. "
        "Set batch_length >= max_workers for optimal parallelization.",
        UserWarning,
    )

In [ ]:
# URL handling (uncomment if needed)
# if isinstance(text_or_documents, str) and io.is_url(text_or_documents):
#    text_or_documents = io.download_text_from_url(text_or_documents)

In [ ]:
# ===>  Prompt template
prompt_template = prompting.PromptTemplateStructured(
    description=prompt_description
)
prompt_template

In [ ]:
prompt_template.examples.extend(examples)
prompt_template

In [ ]:
print(
    prompting.QAPromptGenerator(
        prompt_template,
        format_type=data.FormatType.JSON,
        examples_heading="Ejemplos:\n--------\n",
        question_prefix="Input: ",
        answer_prefix="Output: ",
        fence_output=False,
    ).render("Hola")
)

In [ ]:
# # Model
# language_model = None

# if model:
#     language_model = model
#     if fence_output is not None:
#     language_model.set_fence_output(fence_output)
#     if use_schema_constraints:
#     warnings.warn(
#         "'use_schema_constraints' is ignored when 'model' is provided. "
#         "The model should already be configured with schema constraints.",
#         UserWarning,
#         stacklevel=2,
#     )
# elif config:
#     if use_schema_constraints:
#     warnings.warn(
#         "With 'config', schema constraints are still applied via examples. "
#         "Or pass explicit schema in config.provider_kwargs.",
#         UserWarning,
#         stacklevel=2,
#     )

#     language_model = factory.create_model(
#         config=config,
#         examples=prompt_template.examples if use_schema_constraints else None,
#         use_schema_constraints=use_schema_constraints,
#         fence_output=fence_output,
#     )
# else:
#     if language_model_type != inference.GeminiLanguageModel:
#     warnings.warn(
#         "'language_model_type' is deprecated and will be removed in v2.0.0. "
#         "Use model, config, or model_id parameters instead.",
#         DeprecationWarning,
#         stacklevel=2,
#     )

#     base_lm_kwargs: dict[str, Any] = {
#         "api_key": api_key,
#         "format_type": format_type,
#         "temperature": temperature,
#         "model_url": model_url,
#         "base_url": model_url,
#         "max_workers": max_workers,
#     }

#     # TODO(v2.0.0): Remove gemini_schema parameter
#     if "gemini_schema" in (language_model_params or {}):
#     warnings.warn(
#         "'gemini_schema' is deprecated. Schema constraints are now "
#         "automatically handled. This parameter will be ignored.",
#         DeprecationWarning,
#         stacklevel=2,
#     )
#     language_model_params = dict(language_model_params or {})
#     language_model_params.pop("gemini_schema", None)

#     base_lm_kwargs.update(language_model_params or {})
#     filtered_kwargs = {k: v for k, v in base_lm_kwargs.items() if v is not None}
#     config = factory.ModelConfig(
#         model_id=model_id, provider_kwargs=filtered_kwargs
#     )

#     language_model = factory.create_model(
#         config=config,
#         examples=prompt_template.examples if use_schema_constraints else None,
#         use_schema_constraints=use_schema_constraints,
#         fence_output=fence_output,
#     )

# fence_output = language_model.requires_fence_output

In [ ]:
#model_id = "deepseek-r1:14b"
model_id = "gpt-4o"
language_model_params = {"num_ctx": 12288}

use_schema_constraints = True  # False
fence_output = False
format_type = data.FormatType.JSON

base_lm_kwargs = {
    "api_key": openai_api_key,
    "format_type": format_type,
    "temperature": 0,
    "model_url": "http://localhost:11434",
    "max_workers": max_workers,
}

base_lm_kwargs.update(language_model_params or {})
filtered_kwargs = {k: v for k, v in base_lm_kwargs.items() if v is not None}
config = factory.ModelConfig(
    model_id=model_id, provider_kwargs=filtered_kwargs
)

language_model = factory.create_model(
    config=config,
    examples=prompt_template.examples if use_schema_constraints else None,
    use_schema_constraints=use_schema_constraints,
    fence_output=fence_output,
)

fence_output = language_model.requires_fence_output


In [ ]:
# ===> Resolver

resolver_params = None

resolver_defaults = {
    "fence_output": fence_output,
    "format_type": format_type,
    "extraction_attributes_suffix": "_attributes",
    "extraction_index_suffix": None,
}
resolver_defaults.update(resolver_params or {})

res = resolver.Resolver(**resolver_defaults)
res

In [ ]:
# ===> Annotator
annotator = annotation.Annotator(
    language_model=language_model,
    prompt_template=prompt_template,
    format_type=format_type,
    fence_output=fence_output,
)

# Update prompt generator with custom settings
annotator._prompt_generator = prompting.QAPromptGenerator(
    prompt_template,
    format_type=data.FormatType.JSON,
    examples_heading="Ejemplos:\n--------\n",
    question_prefix="Input: ",
    answer_prefix="Output: ",
    fence_output=False,
)

In [ ]:
prompting.QAPromptGenerator(
    prompt_template,
    format_type=data.FormatType.JSON,
    examples_heading="Ejemplos:\n--------\n",
    question_prefix="Input: ",
    answer_prefix="Output: ",
    fence_output=False,
)

In [ ]:
# Annotation run
# if isinstance(text_or_documents, str):
#     return annotator.annotate_text(
#         text=text_or_documents,
#         resolver=res,
#         max_char_buffer=max_char_buffer,
#         batch_length=batch_length,
#         additional_context=additional_context,
#         debug=debug,
#         extraction_passes=extraction_passes,
#         max_workers=max_workers,
#     )
# else:
#     documents = cast(Iterable[data.Document], text_or_documents)
#     return annotator.annotate_documents(
#         documents=documents,
#         resolver=res,
#         max_char_buffer=max_char_buffer,
#         batch_length=batch_length,
#         debug=debug,
#         extraction_passes=extraction_passes,
#         max_workers=max_workers,
#     )

In [ ]:
text = """
    JUZGADO DE FAMILIA N.º 1 DE LA CIUDAD DE SAN LORENZO
    Expediente N.º 3187/2023
    Carátula: Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar
    SENTENCIA
    En la ciudad de San Lorenzo, Provincia de Santa Fe, a los 22 días del mes de noviembre de 2023, siendo las 09:15 horas, la Sra. Jueza de Familia Dra. Verónica Salvatierra dicta la presente resolución en los autos caratulados “Rodríguez, Ana Carolina c/ Fernández, Diego Esteban s/ Violencia Familiar”, Expte. N.º 3187/2023.
"""

In [ ]:
annotator.annotate_text(
    text,
    resolver=res,
    max_char_buffer=1024,
    **{"timeout": 600, "keep_alive": "10m"},
)

### `langextract.resolver.Resolver()`


```
res = resolver.Resolver(**resolver_defaults)
```
Resolver for YAML/JSON-based **information extraction**.

By default, extractions are returned in the order they appear in the model
output. 

To enable index-based sorting, set **extraction_index_suffix** to a
value like "_index" (the DEFAULT_INDEX_SUFFIX constant). This will sort
extractions by fields ending with that suffix (e.g., "entity_index").

Uses FormatHandler for parsing model output into extractions.

In [ ]:
res = resolver.Resolver(**resolver_defaults)

**Resolver** class can:

- **.resolve(text)** ->  Return the annotated text in the form of a sequence of data.Extraction objects.

- **.align(text,extractions,..)** -> Aligns annotated extractions with source text. Inputs: 

    This uses WordAligner which is based on Python's difflib SequenceMatcher to match tokens in the source text with tokens from the aannotated extractions. If the extraction order is significantly different from the source text order, difflib may skip some matches, leaving certain extractions unmatched. Inputs - > 

    extractions: Annotated extractions. 

    source_text: The text chunk in which to align the extractions. 

    token_offset: The starting token index of the chunk. 

    char_offset: The starting character index of the chunk.

    enable_fuzzy_alignment: Whether to enable fuzzy alignment fallback.

    fuzzy_alignment_threshold: Minimum overlap ratio required for fuzzy alignment.

    accept_match_lesser: Whether to accept partial exact matches (MATCH_LESSER status). **kwargs: Additional parameters.


- **.extract_ordered_extractions(extraction_data)** -> Extracts and orders extraction data based on their associated indexes.

    This function processes a list of dictionaries, each containing pairs of
    extraction class keys and their corresponding values, along with optionally
    associated index keys (identified by the index_suffix). It sorts these pairs
    by their indices in ascending order and excludes pairs without an index key,
    returning a list of lists of tuples (extraction_class: str, extraction_text:
    str).

    Args:
        extraction_data: A list of dictionaries. Each dictionary contains pairs
          of extraction class keys and their values, along with optional index
          keys.

    Returns:
        Extractions sorted by the index attribute or by order of appearance. If
        two
        extractions have the same index, their group order dictates the sorting
        order.




## `langextract.annotation.Annotator` review

```
annotator = annotation.Annotator(
    language_model=language_model, 
    prompt_template=prompt_template, 
    format_type=format_type,
    fence_output=fence_output,)
```

**Annotator** class-> annotates documents with extractions using a language model


### `annotate_text()`

Function **annotate_text()** -> Annotates text with NLP extractions for text input. Returns resolved annotations from text for document.

```
annotator.annotate_text(
    text,
    resolver=res,
    max_char_buffer=1024,
    **{"timeout": 600, "keep_alive": "10m"},
)
```

**text** -> The main text to annotate (resoluciones)

**resolver** = res -> Resolver to use for extracting information from text

**max_char_buffer** -> Max number of characters that we can run inference on. The text will be broken into chunks up to this length.

**extraction_passes** (TO CHECK)-> "Number of sequential extraction passes to improve recall by finding additional entities". We can see if it only detects new entities or if it also correct false positives and reasignes them.


**max_char_buffer** -> Max number of characters that we can run inference on.  The text will be broken into chunks up to this length.

**batch_length** -> Number of chunks to process in a single batch.

**additional_context** -> Additional context to supplement prompt instructions.

**debug** -> Whether to populate debug fields.

**show_progress** -> Whether to show progress bar. Defaults to True.


**We are going to do what annotate_text() function does but without using it.**

In case of debug = True, it tracks time calculating start_time:

In [ ]:
import time
start_time = time.time() if debug else None
start_time

Create a list of documents data.Document() based of texts, the text/texts are given as list of data.Document:

In [ ]:
documents = [data.Document(text=text, document_id=None)]


#### `annotate_documents()`

The function **annotate_documents()** -> Annotates a sequence of documents with NLP extractions.

Breaks documents into chunks, processes them into prompts and performs batched inference, mapping annotated extractions back to the original document. Batch processing is determined by batch_length, and can operate across documents for optimized throughput.

```
annotator.annotate_documents(
        documents,
        resolver = res,
        max_char_buffer=1024,
        debug = False,
        batch_length,
        extraction_passes=1)
        )
```


**documents** -> Documents to annotate. Each document is expected to have a unique document_id. It is a list of data.Document

**resolver** -> Resolver to use for extracting information from text, using resolver.Resolver()

**max_char_buffer** -> Max number of characters that we can run inference on. The text will be broken into chunks up to this length

**batch_length** -> Number of chunks to process in a single batch

**debug** -> Whether to populate debug fields

**extraction_passes** -> Number of sequential extraction attempts to improve recall by finding additional entities. Defaults to 1, which performs standard single extraction. Values > 1 reprocess tokens multiple times, potentially increasing costs with the potential for a more thorough extraction

**show_progress**: Whether to show progress bar. Defaults to True


In [ ]:
documents = [data.Document(text=text, document_id=None)] # documents are redefined in the next cell; they are included here as well for illustrative clarity.

 
annotations = list(
    annotator.annotate_documents(
        documents,
        resolver = res,
        max_char_buffer=60,
        debug = False,
        #batch_length,
        extraction_passes=1)
        )
annotations

In [ ]:
##################################################################### start annotate_documents() code #####################################################################

We introduce the expected inputs of annotate_documents() as variables:

In [ ]:
from langextract import resolver as resolver_lib


# If there are multiple texts, use:
# documents = [data.Document(text=t, document_id=None) for t in texts]

documents = [data.Document(text=text, document_id=None)]
resolver = res
max_char_buffer=200                    # 200 is the default value
debug = False                           # True is the default value
batch_length = 1                        # Default value
extraction_passes = 1                   # Default value
show_progress = False                   # Default value is True, but this is a feature not avaiable in every lx version

In case resolver is None, the function creates one with resolver_lib.Resolver(). We have already defined it.

In [ ]:
if resolver is None:
    resolver = resolver_lib.Resolver(format_type=data.FormatType.YAML)

The function annotate_documents() chooses between to main functions based on extraction_passes value, wich is the number of sequential extraction attempts to improve recall by finding additional entities. 

**extraction_passes == 1** ->  performs standard single extraction. 

**extraction_passes > 1**  ->  reprocess tokens multiple times, potentially increasing costs with the potential for a more thorough extraction.

In [ ]:
if extraction_passes == 1:
    print("Single pass annotation")
    annotations = annotator._annotate_documents_single_pass(
        documents,
        resolver,
        max_char_buffer,
        batch_length,
        debug,
        show_progress=show_progress
    )
else:
    annotations = annotator._annotate_documents_sequential_passes(
        documents,
        resolver,
        max_char_buffer,
        batch_length,
        debug,
        extraction_passes,
        show_progress=show_progress
    )

We will now replicate the functionality of __annotate_documents_single_pass() directly, without calling the function.

#### `Annotator._annotate_documents_single_pass()`

When extraction_passes == 1, **_annotate_documents_single_pass()** is the main function inside annotate_documents() 


```
annotations = annotator._annotate_documents_single_pass(
        documents,
        resolver,
        max_char_buffer,
        batch_length,
        debug,
        show_progress=show_progress
    )
```

Inputs:

**documents** ->  Iterable[data.Document]

**resolver** -> resolver_lib.AbstractResolver

**max_char_buffer** -> int

**batch_length** -> int

**debug** -> bool

**show_progress** -> bool = True

Returns -> Iterator[data.AnnotatedDocument]:

Since all the required inputs have already been defined above, we can proceed directly to the function definition.

In [ ]:
import itertools
from absl import logging

logging.info("Starting document annotation.")
doc_iter, doc_iter_for_chunks = itertools.tee(documents, 2)
curr_document = next(doc_iter, None)


if curr_document is None:
    logging.warning("No documents to process.")
    print("No documents to process.")
    annotations = None  # At this point, the function returns nothing, i.e., "return"


Then it defines chunk_iter based on annotation._document_chunk_iterator:

In [ ]:
class DocumentRepeatError(exceptions.LangExtractError):
  """Exception raised when identical document ids are present."""

#### `_document_chunk_iterator() function`

The return value of this function is determined using chunking.ChunkIterator(). We will first define the function, and then explore the ChunkIterator class in more detail.

In [ ]:
def _document_chunk_iterator(
    documents: Iterable[data.Document],
    max_char_buffer: int,
    restrict_repeats: bool = True,
) -> Iterator[chunking.TextChunk]:
  """Iterates over documents to yield text chunks along with the document ID.

  Args:
    documents: A sequence of Document objects.
    max_char_buffer: The maximum character buffer size for the ChunkIterator.
    restrict_repeats: Whether to restrict the same document id from being
      visited more than once.

  Yields:
    TextChunk containing document ID for a corresponding document.

  Raises:
    DocumentRepeatError: If restrict_repeats is True and the same document ID
      is visited more than once. Valid documents prior to the error will be
      returned.
  """
  visited_ids = set()
  for document in documents:
    tokenized_text = document.tokenized_text
    document_id = document.document_id
    if restrict_repeats and document_id in visited_ids:
      raise DocumentRepeatError(
          f"Document id {document_id} is already visited."
      )

    # This ChunkIterator iterates  through chunks of a tokenized text. We will explore the ChunkIterator class in more detail below.
    chunk_iter = chunking.ChunkIterator(
        text=tokenized_text,
        max_char_buffer=max_char_buffer,
        document=document,
    )
    
    visited_ids.add(document_id)

    return chunk_iter

`chunking.ChunkIterator() class`

**ChunkIterator** class iterate through chunks of a tokenized text. Chunks may consist of sentences or sentence fragments that can fit into the maximum character buffer that we can run inference on.

  A)
  If a sentence length exceeds the max char buffer, then it needs to be broken
  into chunks that can fit within the max char buffer. We do this in a way that
  maximizes the chunk length while respecting newlines (if present) and token
  boundaries.
  Consider this sentence from a poem by John Donne:

  ```
  No man is an island,
  Entire of itself,
  Every man is a piece of the continent,
  A part of the main.
  ```
  With max_char_buffer=40, the chunks are:
  * "No man is an island,\nEntire of itself," len=38
  * "Every man is a piece of the continent," len=38
  * "A part of the main." len=19

  B)
  If a single token exceeds the max char buffer, it comprises the whole chunk.
  Consider the sentence:
  "This is antidisestablishmentarianism."
  With max_char_buffer=20, the chunks are:
  * "This is" len=7
  * "antidisestablishmentarianism" len=28
  * "." len(1)

  C)
  If multiple *whole* sentences can fit within the max char buffer, then they
  are used to form the chunk.
  Consider the sentences:
  "Roses are red. Violets are blue. Flowers are nice. And so are you."
  With max_char_buffer=60, the chunks are:
  * "Roses are red. Violets are blue. Flowers are nice." len=50
  * "And so are you." len=15
  

In [ ]:
annotated_extractions: list[data.Extraction] = []

So, we now apply _document_chunk_iterator() on documentos to chunk them to create text chunks along with the document ID.

In [ ]:
doc_iter_for_chunks

In [ ]:
chunk_iter = _document_chunk_iterator(doc_iter_for_chunks, max_char_buffer) 
# or chunk_iter = annotator._document_chunk_iterator(documents, max_char_buffer) if we want to use the annotator class


In [ ]:
chunk_iter

In [ ]:
import more_itertools

Now we processes chunks into batches of TextChunk for inference

In [ ]:
def make_batches_of_textchunk(
    chunk_iter: Iterator[chunking.TextChunk],
    batch_length: int,
) -> Iterable[Sequence[chunking.TextChunk]]:
  """Processes chunks into batches of TextChunk for inference, using itertools.batched.

  Args:
    chunk_iter: Iterator of TextChunks.
    batch_length: Number of chunks to include in each batch.

  Yields:
    Batches of TextChunks.
  """
  for batch in more_itertools.batched(chunk_iter, batch_length):
    return list(batch)

In [ ]:
chunk_iter

In [ ]:
batches = make_batches_of_textchunk(chunk_iter, batch_length)  # or chunking.make_batches_of_textchunk(chunk_iter, batch_length)
batches

The function **get_model_info()** extracts model information from _annotate_documents_single_pass._language_model if the function _annotate_documents_single_pass receives _language_model as input (which should include either model_id or model_url information).

In [ ]:
from typing import Any

_language_model = ''

def get_model_info(language_model: Any) -> str | None:
  """Extract model information from a language model instance.

  Args:
    language_model: A language model instance.

  Returns:
    A string describing the model, or None if not available.
  """
  if hasattr(language_model, "model_id"):
    return language_model.model_id

  if hasattr(language_model, "model_url"):
    return language_model.model_url

  return None
  

In [ ]:
import tqdm
# ANSI color codes for terminal output
BLUE = "\033[94m"
GREEN = "\033[92m"
CYAN = "\033[96m"
BOLD = "\033[1m"
RESET = "\033[0m"

# Google Blue color for progress bars
GOOGLE_BLUE = "#4285F4"

def format_extraction_progress(
    model_info: str | None,
    current_chars: int | None = None,
    processed_chars: int | None = None,
) -> str:
  """Format the complete extraction progress bar description.

  Args:
    model_info: Optional model information (e.g., "gemini-2.0-flash").
    current_chars: Number of characters in current batch (optional).
    processed_chars: Total number of characters processed so far (optional).

  Returns:
    Formatted description string.
  """
  # Base description
  if model_info:
    desc = f"{BLUE}{BOLD}LangExtract{RESET}: model={GREEN}{model_info}{RESET}"
  else:
    desc = f"{BLUE}{BOLD}LangExtract{RESET}: Processing"

  # Add stats if provided
  if current_chars is not None and processed_chars is not None:
    current_str = f"{GREEN}{current_chars:,}{RESET}"
    processed_str = f"{GREEN}{processed_chars:,}{RESET}"
    desc += f", current={current_str} chars, processed={processed_str} chars"

  return desc

def create_extraction_progress_bar(
    iterable: Any, model_info: str | None = None, disable: bool = False
) -> tqdm.tqdm:
  """Create a styled progress bar for extraction.

  Args:
    iterable: The iterable to wrap with progress bar.
    model_info: Optional model information to display (e.g., "gemini-1.5-pro").
    disable: Whether to disable the progress bar.

  Returns:
    A configured tqdm progress bar.
  """
  desc = format_extraction_progress(model_info)

  return tqdm.tqdm(
      iterable,
      desc=desc,
      bar_format="{desc} [{elapsed}]",
      disable=disable,
      dynamic_ncols=True,
  )

In [ ]:
# import progress
# model_info= progress.get_model_info(_annotate_documents_single_pass.get_language_model)

from langextract import progress

model_info = get_model_info(_language_model)
progress_bar = progress.create_extraction_progress_bar( #create_extraction_progress_bar(
    batches, model_info=model_info, disable=not show_progress
)

In [ ]:
logging.info("Starting document annotation.")

doc_iter, doc_iter_for_chunks = itertools.tee(documents, 2)
curr_document = next(doc_iter, None)
annotations = []
if curr_document is None:
    logging.warning("No documents to process.")
else:
    annotated_extractions = []
    chunk_iter = _document_chunk_iterator(doc_iter_for_chunks, max_char_buffer)
    batches = chunking.make_batches_of_textchunk(chunk_iter, batch_length)
    model_info = progress.get_model_info(_language_model)
    progress_bar = progress.create_extraction_progress_bar(
        batches, model_info=model_info, disable=not show_progress
    )
    chars_processed = 0
    for index, batch in enumerate(progress_bar):
        logging.info("Processing batch %d with length %d", index, len(batch))
        batch_prompts = []
        for text_chunk in batch:
            batch_prompts.append(
                annotator._prompt_generator.render(
                    question=text_chunk.chunk_text,
                    additional_context=text_chunk.additional_context,
                )
            )
        # Show what we're currently processing
        if debug and progress_bar:
            batch_size = sum(len(chunk.chunk_text) for chunk in batch)
            desc = progress.format_extraction_progress(
                model_info,
                current_chars=batch_size,
                processed_chars=chars_processed,
            )
            progress_bar.set_description(desc)

        batch_scored_outputs = annotator._language_model.infer(
            batch_prompts=batch_prompts,
        )

        # Update total processed
        if debug:
            for chunk in batch:
                if hasattr(chunk, "document_text") and chunk.document_text:
                    char_interval = chunk.char_interval
                    chars_processed += char_interval.end_pos - char_interval.start_pos
            if progress_bar:
                batch_size = sum(len(chunk.chunk_text) for chunk in batch)
                desc = progress.format_extraction_progress(
                    model_info,
                    current_chars=batch_size,
                    processed_chars=chars_processed,
                )
                progress_bar.set_description(desc)

        for text_chunk, scored_outputs in zip(batch, batch_scored_outputs):
            logging.debug("Processing chunk: %s", text_chunk)
            if not scored_outputs:
                logging.error(
                    "No scored outputs for chunk with ID %s.", text_chunk.document_id
                )
                raise exceptions.InferenceOutputError(
                    "No scored outputs from language model."
                )
            while curr_document.document_id != text_chunk.document_id:
                logging.info(
                    "Completing annotation for document ID %s.",
                    curr_document.document_id,
                )
                annotated_doc = data.AnnotatedDocument(
                    document_id=curr_document.document_id,
                    extractions=annotated_extractions,
                    text=curr_document.text,
                )
                annotations.append(annotated_doc)
                annotated_extractions.clear()
                curr_document = next(doc_iter, None)
                assert curr_document is not None, (
                    f"Document should be defined for {text_chunk} per"
                    " _document_chunk_iterator(...) specifications."
                )

            top_inference_result = scored_outputs[0].output
            logging.debug("Top inference result: %s", top_inference_result)

            annotated_chunk_extractions = resolver.resolve(
                top_inference_result, debug=debug
            )
            chunk_text = text_chunk.chunk_text
            token_offset = text_chunk.token_interval.start_index
            char_offset = text_chunk.char_interval.start_pos

            aligned_extractions = resolver.align(
                annotated_chunk_extractions,
                chunk_text,
                token_offset,
                char_offset,
            )
            annotated_extractions.extend(aligned_extractions)

    progress_bar.close()

    if debug:
        progress.print_extraction_complete()

    if curr_document is not None:
        logging.info(
            "Finalizing annotation for document ID %s.", curr_document.document_id
        )
        annotated_doc = data.AnnotatedDocument(
            document_id=curr_document.document_id,
            extractions=annotated_extractions,
            text=curr_document.text,
        )
        annotations.append(annotated_doc)

    logging.info("Document annotation completed.")


In [ ]:
##################################################################### end annotate_documents() code #####################################################################

Verify that exactly one annotation (one document) was produced before continuing.

If not, raise an AssertionError showing the actual count to aid debugging.

In [ ]:
assert len(annotations) == 1, f"Expected 1 annotation but got {len(annotations)} annotations."

In [ ]:
max_char_buffer = 1024
from langextract import progress

In [ ]:
if debug and annotations[0].extractions:
     elapsed_time = time.time() - start_time if start_time else None
     num_extractions = len(annotations[0].extractions)
     unique_classes = len(
         set(e.extraction_class for e in annotations[0].extractions)
     )
     num_chunks = len(text) // max_char_buffer + (
         1 if len(text) % max_char_buffer else 0
     )

     progress.print_extraction_summary(
         num_extractions,
         unique_classes,
         elapsed_time=elapsed_time,
         chars_processed=len(text),
         num_chunks=num_chunks,
     )


In [ ]:
annotations

In [ ]:
data.AnnotatedDocument(
     document_id=annotations[0].document_id,
     extractions=annotations[0].extractions,
     text=annotations[0].text,
 )